In [5]:
# Load CSV

import pandas as pd

df = pd.read_csv("know_disc_assign2.csv")


In [6]:
# Convert 

cat_cols = ["age_group", "race_group", "birthplace_group"]
for col in cat_cols:
    df[col] = df[col].astype("category")


In [7]:
df_encoded = pd.get_dummies(df, columns=cat_cols)


In [8]:
X = df_encoded.drop(columns=["moved"])
y = df_encoded["moved"]


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [10]:
from sklearn.naive_bayes import BernoulliNB

model = BernoulliNB()
model.fit(X_train, y_train)


BernoulliNB()

In [11]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.6163566804777677

Confusion Matrix:
 [[2362374 1264351]
 [1385938 1895548]]

Classification Report:
               precision    recall  f1-score   support

           0       0.63      0.65      0.64   3626725
           1       0.60      0.58      0.59   3281486

    accuracy                           0.62   6908211
   macro avg       0.62      0.61      0.61   6908211
weighted avg       0.62      0.62      0.62   6908211



In [12]:
import numpy as np

feature_names = X.columns
log_probs = model.feature_log_prob_

for i, cls in enumerate(model.classes_):
    print(f"\nClass {cls} log probabilities:")
    top_features = np.argsort(log_probs[i])[::-1][:10]
    for idx in top_features:
        print(f"{feature_names[idx]}: {log_probs[i][idx]}")



Class 0 log probabilities:
year: -6.89614942928074e-08
urban_status: -6.89614942928074e-08
race_group_White: -0.01426911540493947
birthplace_group_US: -0.5594713068036601
sex_binary: -0.6998754423583584
age_group_0_14: -0.8164836122875183
birthplace_group_Unknown: -1.0649294929834774
age_group_25_44: -1.4902715010588619
age_group_15_24: -1.5827257360305858
age_group_45_64: -2.289543410473401

Class 1 log probabilities:
year: -7.61498775148084e-08
urban_status: -7.61498775148084e-08
race_group_White: -0.008854115575470445
sex_binary: -0.5413112401665501
birthplace_group_US: -0.5979132781480789
birthplace_group_Unknown: -0.9040846870924639
age_group_25_44: -1.0933484476657167
age_group_0_14: -1.420627439568639
age_group_45_64: -1.5880872043842995
age_group_15_24: -1.6857784180549942


In [16]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.naive_bayes import BernoulliNB

model = BernoulliNB()

kf = KFold(n_splits=10, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y, cv=kf, scoring='accuracy')

print("10-fold CV accuracy:", scores.mean())
print("Fold-by-fold accuracy:", scores)


10-fold CV accuracy: 0.6162682484615292
Fold-by-fold accuracy: [0.6164779  0.61623575 0.61643291 0.61606697 0.61654408 0.61637327
 0.61574503 0.61653974 0.61631798 0.61594885]


In [ ]:
# to big locked up

'''
import numpy as np
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import BernoulliNB
from sklearn.dummy import DummyClassifier

# X = your feature matrix
# y = your target variable (movement: 0 or 1)

kf = KFold(n_splits=10, shuffle=True, random_state=42)

# ZeroR (majority class predictor)
zeroR = DummyClassifier(strategy="most_frequent")
zeroR_scores = cross_val_score(zeroR, X, y, cv=kf)

# Logistic Regression
log_reg = LogisticRegression(max_iter=1000)
log_scores = cross_val_score(log_reg, X, y, cv=kf)

# Naive Bayes
nb = BernoulliNB()
nb_scores = cross_val_score(nb, X, y, cv=kf)

print("ZeroR Accuracy: {:.4f} ± {:.4f}".format(zeroR_scores.mean(), zeroR_scores.std()))
print("Logistic Regression Accuracy: {:.4f} ± {:.4f}".format(log_scores.mean(), log_scores.std()))
print("Naive Bayes Accuracy: {:.4f} ± {:.4f}".format(nb_scores.mean(), nb_scores.std()))
'''

In [13]:
from sklearn.model_selection import train_test_split

# Take a 3% stratified sample
X_sample, _, y_sample, _ = train_test_split(
    X, y, 
    test_size=0.97, 
    stratify=y, 
    random_state=42
)

print("Sample size:", X_sample.shape)


Sample size: (1036231, 20)


In [18]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import BernoulliNB


kf = KFold(n_splits=10, shuffle=True, random_state=42)

zeroR = DummyClassifier(strategy="most_frequent")
zeroR_scores = cross_val_score(zeroR, X_sample, y_sample, cv=kf)

log_reg = LogisticRegression(max_iter=1000)
log_scores = cross_val_score(log_reg, X_sample, y_sample, cv=kf)

nb = BernoulliNB()
nb_scores = cross_val_score(nb, X_sample, y_sample, cv=kf)

print("ZeroR:", zeroR_scores.mean(), zeroR_scores.std())
print("Logistic:", log_scores.mean(), log_scores.std())
print("Naive Bayes:", nb_scores.mean(), nb_scores.std())


ZeroR: 0.5248125177483256 0.0011644455881769592
Logistic: 0.6258556252066009 0.00132948883089013
Naive Bayes: 0.6200876015341763 0.004795742984250069


In [19]:
from scipy.stats import ttest_rel

t_lr_nb = ttest_rel(log_scores, nb_scores)
t_lr_zero = ttest_rel(log_scores, zeroR_scores)
t_nb_zero = ttest_rel(nb_scores, zeroR_scores)

print("LR vs NB:", t_lr_nb)
print("LR vs ZeroR:", t_lr_zero)
print("NB vs ZeroR:", t_nb_zero)


LR vs NB: Ttest_relResult(statistic=3.5749056615177257, pvalue=0.005978187670260017)
LR vs ZeroR: Ttest_relResult(statistic=175.5519969881547, pvalue=3.211442102802446e-17)
NB vs ZeroR: Ttest_relResult(statistic=55.23209060941358, pvalue=1.051752379088626e-12)


In [20]:
print(df.dtypes)


year                   int64
sex_binary             int64
age_group           category
race_group          category
birthplace_group    category
urban_status           int64
gq_binary              int64
blind                  int64
deaf                   int64
IDIOTIC                int64
insane                 int64
moved                  int64
dtype: object
